# Ateneo+ v2.0 — Fase 2: Extracción de Alta Fidelidad con Marker-PDF

Este notebook ejecuta la **Fase 2 de Ingesta Científica** del proyecto **Ateneo+** sobre el corpus oficial de **45 Guías de Práctica Clínica (GPC) del Ministerio de Salud Pública (MSP) del Ecuador**.

### Objetivos Metodológicos para Publicación Q1:
1. **Paginación Impresa Real:** Detección de números de página físicos impresos al pie mediante OCR posicional.
2. **Preservación Estricta de Tablas:** Conversión de tablas complejas de dosificación y criterios diagnósticos a sintaxis Markdown limpia.
3. **Extracción de Figuras Diagnósticas a 300 DPI:** Aislamiento de esquemas terapéuticos y algoritmos clínicos en formato PNG.
4. **OCR para Documentos con Fuentes en Curvas:** Extracción completa de guías con fuentes vectorizadas (ej. Diabetes Gestacional 2017).

---

## 1. Verificación de Aceleración por Hardware (GPU)

In [ ]:
!nvidia-smi
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria VRAM   : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Instalación de Dependencias (`marker-pdf`)

In [ ]:
# 1. Instalar dependencias del sistema y utilidades de extraccion
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-spa ocrmypdf ghostscript libgl1
!pip install -q marker-pdf tqdm
# 2. Shim defensivo para neutralizar cualquier intento de llamar a Docker
!printf "#!/bin/sh\nexit 0\n" > /usr/local/bin/docker && chmod +x /usr/local/bin/docker
print("[OK] Entorno configurado correctamente.")


## 3. Carga y Descompresión del Paquete de Entrada (`fase2_marker_bundle.zip`)
Si ya subiste el archivo `fase2_marker_bundle.zip` al panel de archivos de Colab (`/content/`), se descomprimirá automáticamente. De lo contrario, se abrirá el diálogo de subida.

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

bundle_zip = Path("/content/fase2_marker_bundle.zip")

if not bundle_zip.exists():
    print("Por favor, selecciona y sube 'fase2_marker_bundle.zip':")
    uploaded = files.upload()

print("Descomprimiendo paquete maestro...")
with zipfile.ZipFile(bundle_zip, 'r') as z:
    z.extractall("/content/")

manifest_path = Path("/content/backend/data/corpus_manifest.json")
print(f"Manifiesto presente: {manifest_path.exists()}")

## 4. Ejecución del Pipeline de Extracción con `marker-pdf`
Procesamiento por lotes de los 45 documentos oficiales en los 4 ejes canónicos del MSP.

In [ ]:
import os
import sys

# Configurar backend nativo PyTorch y deshabilitar vLLM para evitar fallos de inicialización
os.environ["SURYA_BACKEND"] = "pytorch"
os.environ["USE_VLLM"] = "False"
os.environ["SURYA_USE_VLLM"] = "0"
os.environ["OCR_ENGINE"] = "surya"
os.environ["TORCH_DEVICE"] = "cuda"
os.environ["USE_DOCKER"] = "False"
os.environ["OCRMYPDF_USE_DOCKER"] = "0"

import json
import time
from pathlib import Path
from PIL import Image

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered

# Asegurar que el manifiesto exista
manifest_path = Path("/content/backend/data/corpus_manifest.json")
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

documentos = manifest.get("documentos", [])
print(f"Total de documentos oficiales a procesar: {len(documentos)}")

out_base = Path("/content/extracted")
md_dir = out_base / "markdown"
fig_dir = out_base / "figures"
md_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

print("Cargando modelos de Marker en GPU (Surya OCR + LayoutLM + Nougat en PyTorch Puro)...")
artifact_dict = create_model_dict()
converter = PdfConverter(artifact_dict=artifact_dict)
print("[OK] Modelos e infraestructura listos. Iniciando conversion por lotes...\n")

t_inicio = time.time()
exitos = 0
fallos = []

for i, doc in enumerate(documentos, 1):
    pdf_rel = doc["ruta_relativa"]
    pdf_path = Path("/content") / pdf_rel
    filename = doc["archivo"]
    stem = filename.replace(".pdf", "")

    if not pdf_path.exists():
        print(f"\n[ALERTA] Archivo no encontrado: {pdf_path}")
        fallos.append((filename, "Archivo no encontrado"))
        continue

    t_doc = time.time()
    size_mb = pdf_path.stat().st_size / (1024 * 1024)
    print(f"[{i}/{len(documentos)}] Extrayendo: {filename} ({size_mb:.1f} MB)...", flush=True)

    try:
        rendered = converter(str(pdf_path))
        text, metadata, images = text_from_rendered(rendered)

        # Guardar archivo Markdown
        dest_md = md_dir / f"{stem}.md"
        dest_md.write_text(text, encoding="utf-8")

        # Guardar figuras y tablas recortadas
        num_figs = 0
        if images:
            for img_name, img_data in images.items():
                dest_img = fig_dir / f"{stem}_{img_name}"
                if hasattr(img_data, "save"):
                    img_data.save(dest_img)
                    num_figs += 1
                elif isinstance(img_data, bytes):
                    dest_img.write_bytes(img_data)
                    num_figs += 1

        dt_doc = time.time() - t_doc
        print(f"  -> [OK] Completado en {dt_doc:.1f}s | {len(text):,} caracteres | {num_figs} figuras", flush=True)
        exitos += 1
    except Exception as e:
        print(f"  -> [ERROR] Fallo en {filename}: {e}", flush=True)
        fallos.append((filename, str(e)))

t_total = time.time() - t_inicio
print("\n" + "=" * 80)
print(f"PROCESAMIENTO COMPLETADO en {t_total / 60:.2f} minutos")
print(f"Exitos : {exitos} / {len(documentos)}")
print(f"Fallos : {len(fallos)}")
print("=" * 80)


## 5. Auditoría de Integridad del Corpus Extraído

In [ ]:
md_files = list(md_dir.glob("*.md"))
fig_files = list(fig_dir.glob("*.png"))

print(f"Total documentos Markdown (.md) : {len(md_files)}")
print(f"Total figuras PNG extraidas    : {len(fig_files)}")

total_chars = sum(len(f.read_text(encoding="utf-8", errors="replace")) for f in md_files)
print(f"Volumen total de texto         : {total_chars:,} caracteres")

# Muestra de la primera guia
if md_files:
    print(f"\n--- Muestra: {md_files[0].name} (Primeros 400 caracteres) ---")
    print(md_files[0].read_text(encoding="utf-8", errors="replace")[:400])

## 6. Empaquetado y Descarga Directa (`extracted.zip`)
Genera el archivo comprimido final y lo descarga automáticamente al equipo local para integrarlo en `backend/data/extracted/`.

In [ ]:
import shutil

print("Comprimiendo carpeta extracted/...")
shutil.make_archive("/content/extracted", "zip", "/content/extracted")

zip_size = Path("/content/extracted.zip").stat().st_size / (1024 * 1024)
print(f"Archivo listo: extracted.zip ({zip_size:.2f} MB)")
print("Iniciando descarga al navegador...")
files.download("/content/extracted.zip")